# Notebook 03 — Operations & Broadcasting

**numpy-mastery** · Module 03 of 06

> **Goal**: perform arithmetic on whole arrays at once, aggregate data along any axis,
> and understand broadcasting — the rule set that lets arrays of different shapes
> work together without writing a single loop.

**What you'll be able to do after this notebook:**
- Apply arithmetic, comparison, and aggregation operations vectorised
- Control exactly which axis an aggregation collapses
- Understand and apply NumPy's 3 broadcasting rules
- Recognise and use real broadcasting patterns from ML code

---


## 0 · Setup

In [1]:
import numpy as np

---
## 1 · Element-wise arithmetic

When two arrays have the **same shape**, every operator applies element-by-element.


In [3]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print(a + b)
print(b - a)
print(a * b)
print(b / a)
print(b // a)   # floor division
print(b % a)    # modulo
print(a ** 2)   # power

[11 22 33 44]
[ 9 18 27 36]
[ 10  40  90 160]
[10. 10. 10. 10.]
[10 10 10 10]
[0 0 0 0]
[ 1  4  9 16]


### Scalar operations

A scalar is treated as if it were "stretched" to match the array's shape, this is actually the simplest case of broadcasting (Section 4).

In [4]:
a = np.array([1, 2, 3, 4])

print(a * 3)
print(a + 100)
print(1 / a)
print(a > 2)

[ 3  6  9 12]
[101 102 103 104]
[1.         0.5        0.33333333 0.25      ]
[False False  True  True]


---
## 2 · Comparison operators → boolean arrays

Every comparison returns a boolean array, not a single True/False.
This is the foundation for boolean masking from Notebook 02.


In [8]:
a = np.array([3, 7, 1, 9, 4])
b = np.array([5, 5, 5, 5, 5])

print(a > b)
print(a == b)
print(a != b)

# Array-to-array equality check (returns array, NOT a single bool!)
print(np.array_equal(a, b))  # False — used for single bool result
print(np.allclose(a, a + 1e-10)) # True  — for float comparisons with tolerance


[False  True False  True False]
[False False False False False]
[ True  True  True  True  True]
False
True


---
## 3 · Aggregation functions and the `axis` parameter

This is one of the most important concepts in all of NumPy.  
Aggregations *collapse* one or more dimensions down to a single value.

In [11]:
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# no axis → collapse everything to a single scalar

print(f"sum (all) : {M.sum()}")
print(f"mean (all) : {M.mean()}")
print(f"min (all) : {M.min()}")
print(f'max (all) : {M.max()}')
print(f'min (all) : {M.min()}')
print(f'std (all) : {M.std()}')

sum (all) : 45
mean (all) : 5.0
min (all) : 1
max (all) : 9
min (all) : 1
std (all) : 2.581988897471611


### Understanding `axis=0` vs `axis=1`

```
         col0  col1  col2
row0  [   1     2     3  ]
row1  [   4     5     6  ]
row2  [   7     8     9  ]

axis=0  ↓  (collapses ROWS — operates DOWN each column)
        → result has shape (3,)  one value per COLUMN

axis=1  →  (collapses COLUMNS — operates ACROSS each row)
        → result has shape (3,)  one value per ROW
```

**Memory trick**: the axis you specify is the one that *disappears* from the result.

In [13]:
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])


print(f"sum(axis = 0) : {M.sum(axis = 0)}")
print(f"sum(axis = 1) : {M.sum(axis = 1)}")


print("mean(axis=0):", M.mean(axis=0)) 
print("mean(axis=1):", M.mean(axis=1)) 

print("max(axis=0):", M.max(axis=0))  
print("max(axis=1):", M.max(axis=1)) 


sum(axis = 0) : [12 15 18]
sum(axis = 1) : [ 6 15 24]
mean(axis=0): [4. 5. 6.]
mean(axis=1): [2. 5. 8.]
max(axis=0): [7 8 9]
max(axis=1): [3 6 9]


In [14]:
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

no_keep = M.sum(axis =1)
with_keep = M.sum(axis = 1 , keepdims = True)

print(f"without keeping the dim :  {no_keep.shape} \n {no_keep}")
print(f"with keeping the dim : {with_keep.shape} \n {with_keep}")

without keeping the dim :  (3,) 
 [ 6 15 24]
with keeping the dim : (3, 1) 
 [[ 6]
 [15]
 [24]]


---
## 4 · Broadcasting, the 3 rules

Broadcasting lets NumPy operate on arrays of **different shapes** by virtually
"stretching" the smaller one — without actually copying any data.

> **Rule 1**: If the arrays have a different number of dimensions, prepend 1s
> to the smaller shape until they match in length.  
> **Rule 2**: Any dimension of size 1 is stretched to match the other array's
> size along that dimension.  
> **Rule 3**: If two dimensions disagree and neither is 1 → `ValueError`.


### Example 1 — scalar + array (simplest case)

In [15]:
a = np.array([[1, 2, 3], [4, 5, 6]])   # shape (2, 3)
result = a + 10
# 10 is treated as shape () → stretched to (2,3)
print(result)
# [[11 12 13]
#  [14 15 16]]

[[11 12 13]
 [14 15 16]]


### Example 2 — (2,3) + (3,)

```
a shape:    (2, 3)
b shape:       (3,)  →  prepend 1  →  (1, 3)  →  stretch  →  (2, 3)
```

In [16]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])        # shape (2, 3)
b = np.array([10, 20, 30])       # shape (3,)

result = a + b
print(result)
# [[11 22 33]
#  [14 25 36]]
# b is "copied" across both rows — but no actual copy happens in memory

[[11 22 33]
 [14 25 36]]


### Example 3 — (3,1) + (1,3) → outer broadcasting

This is the pattern behind multiplication tables, pairwise distances, and
attention score matrices in transformers.

In [17]:
col = np.array([[0], [10], [20]])    # shape (3, 1)
row = np.array([[1, 2, 3]])          # shape (1, 3)

result = col + row
print(result)
# [[ 1  2  3]
#  [11 12 13]
#  [21 22 23]]
# Every combination of col[i] + row[j] appears — an "outer" operation

[[ 1  2  3]
 [11 12 13]
 [21 22 23]]


### Example 4 — incompatible shapes → error

In [18]:
a = np.array([1, 2])      # shape (2,)
b = np.array([1, 2, 3])   # shape (3,)

try:
    a + b
except ValueError as e:
    print("ValueError:", e)
# neither dimension is 1, and 2 != 3 → cannot broadcast

ValueError: operands could not be broadcast together with shapes (2,) (3,) 


### Checking shapes before broadcasting — `np.broadcast_shapes`

Useful for debugging without actually running the (possibly expensive) operation.

In [19]:
print(np.broadcast_shapes((2, 3), (3,)))      # (2, 3)
print(np.broadcast_shapes((3, 1), (1, 4)))    # (3, 4)
print(np.broadcast_shapes((5, 1, 3), (1, 4, 1))) # (5, 4, 3)

try:
    np.broadcast_shapes((2, 3), (4,))
except ValueError as e:
    print("Error:", e)


(2, 3)
(3, 4)
(5, 4, 3)
Error: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (2, 3) and arg 1 with shape (4,).


In [22]:
try :
    np.broadcast_shapes((4 , 5) , (6))
except ValueError as e:
    print("Error" , e)

Error shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (4, 5) and arg 1 with shape (6,).


---
## 5 · Real broadcasting patterns

These three patterns show up constantly in data science and ML code.

### Pattern 1 — centring data (subtract column mean)

In [25]:
X = np.array([[1.0, 10.0],
              [2.0, 20.0],
              [3.0, 30.0]])

col_mean = X.mean(axis = 0)
print(X.shape)
print(col_mean.shape)

X_centered = X - col_mean

print("column means:", col_mean)
print("centered:\n", X_centered)
print("new column means:", X_centered.mean(axis=0))

(3, 2)
(2,)
column means: [ 2. 20.]
centered:
 [[ -1. -10.]
 [  0.   0.]
 [  1.  10.]]
new column means: [0. 0.]


### Pattern 2 — combining a row vector and column vector (outer sum/product)

In [36]:
# Build a small "distance from edges" grid using broadcasting

rows = np.arange(4)[: ,None]
cols = np.arange(5)[None, :]

print(rows)
print()
print(cols)
print()
grid = rows + cols
print(grid)

[[0]
 [1]
 [2]
 [3]]

[[0 1 2 3 4]]

[[0 1 2 3 4]
 [1 2 3 4 5]
 [2 3 4 5 6]
 [3 4 5 6 7]]


### Pattern 3 — comparing every element of one array to every element of another

In [41]:
labels = np.array([0, 1, 2, 0, 3])
classes = np.arange(4)

# lables [: None] shape (5, 1) vs classes [None, : ] shape (1, 4) → (5,4)
one_hot = (labels[: , None] == classes[None , :]).astype(int)
print(one_hot)

[[1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [1 0 0 0]
 [0 0 0 1]]



---
## 6 · In-place operations

`+=`, `*=`, etc. modify the array in place — faster (no new memory allocated)
but be careful: this can silently mutate data shared elsewhere.


In [45]:
a = np.array([1.0, 2.0, 3.0])
b = a      # be is the same array, not a copy

a += 10
print(a)    # modifies in place
print(b)

# compare with out_of_place
a = np.array([1.0, 2.0, 3.0])
c = a + 10  # creates a new array
print(a)
print(c)


[11. 12. 13.]
[11. 12. 13.]
[1. 2. 3.]
[11. 12. 13.]


> **Watch out**: `arr = arr + 1` and `arr += 1` look identical but behave
> differently if `arr` is shared with another variable. When in doubt about
> shared references, use the out-of-place version.

---
## 7 · Summary cheatsheet

```python
# Element-wise arithmetic (same shape)
a + b,  a - b,  a * b,  a / b,  a ** b,  a // b,  a % b

# Comparisons → boolean array
a > b,  a == b,  a != b
np.array_equal(a, b)      # single bool, exact match
np.allclose(a, b)         # single bool, with tolerance

# Aggregation
arr.sum(), .mean(), .min(), .max(), .std(), .var()
arr.sum(axis=0)            # collapse axis 0 (down columns)
arr.sum(axis=1)            # collapse axis 1 (across rows)
arr.sum(axis=1, keepdims=True)  # keep collapsed dim as size 1

# Broadcasting rules
# 1. Prepend 1s to the smaller shape
# 2. Stretch any dimension of size 1
# 3. Error if dimensions disagree and neither is 1

np.broadcast_shapes(shape1, shape2)  # check compatibility without computing

# In-place vs out-of-place
a += 1     # modifies a (and anything sharing memory with a)
a = a + 1  # creates a new array
```

---
